In [0]:
import pyspark.sql.functions as F

# print("cleansing claims data")

# Read bronze data to clean in silver:

df_claims = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/claims")

# 2. Cleanse and Cast Types

df_claims_clean = df_claims\
    .withColumn("service_date", F.to_date(F.col("service_date"))) \
    .withColumn("admission_date", F.to_date(F.col("admission_date"))) \
    .withColumn("discharge_date", F.to_date(F.col("discharge_date"))) \
    .withColumn("payment_date", F.to_date(F.col("payment_date"))) \
    .withColumn("paid_amount", F.col("paid_amount").cast("double")) \
    .fillna({
        "rejection_reason": "Not Rejected", 
        "paid_amount": 0.0,
        "claim_status": "Unknown"
    })

display(df_claims_clean)

#Writing to silver
df_claims_clean.write.format("delta").mode("overwrite").save("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/claims")


claim_id,beneficiary_id,provider_id,claim_type,service_date,admission_date,discharge_date,diagnosis_code,procedure_code,billed_amount,paid_amount,claim_status,rejection_reason,payment_date,source_file,ingested_at,Bronze_timestamp,Bronze_file_path
C0000001,B00090,P0028,PHYSICIAN,2024-07-12,null,null,I10,99223,589.46,534.48,APPROVED,Not Rejected,2024-09-03,claims_202407.csv,2024-07-14T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000002,B00323,P0016,PHYSICIAN,2025-10-03,null,null,M17.9,85025,1848.0,0.0,REJECTED,NOT_COVERED,null,claims_202510.csv,2025-10-05T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000003,B00053,P0032,PHYSICIAN,2024-11-20,null,null,R07.9,36415,317.17,288.71,APPROVED,Not Rejected,2025-01-12,claims_202411.csv,2024-11-21T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000004,B00106,P0040,PHYSICIAN,2024-01-17,null,null,R07.9,93000,2227.36,1696.57,APPROVED,Not Rejected,2024-02-11,claims_202401.csv,2024-01-19T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000005,B00291,P0016,PHYSICIAN,2025-02-21,null,null,M17.9,99213,1209.03,945.49,APPROVED,Not Rejected,2025-04-01,claims_202502.csv,2025-02-22T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000006,B00385,P0006,OUTPATIENT,2024-10-14,null,null,I25.10,99214,898.23,199.23,PARTIALLY_PAID,Not Rejected,2024-10-31,claims_202410.csv,2024-10-14T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000007,B00377,P0024,OUTPATIENT,2024-12-11,null,null,E78.5,93000,525.48,427.71,APPROVED,Not Rejected,2025-01-02,claims_202412.csv,2024-12-12T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000008,B00064,P0002,PHYSICIAN,2024-07-08,null,null,E11.9,80053,1657.36,1141.49,APPROVED,Not Rejected,2024-07-27,claims_202407.csv,2024-07-09T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000009,B00148,P0013,PHYSICIAN,2024-04-08,null,null,N39.0,93000,2462.58,1419.68,PARTIALLY_PAID,Not Rejected,2024-06-01,claims_202404.csv,2024-04-10T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv
C0000010,B00206,P0006,OUTPATIENT,2024-02-22,null,null,R07.9,93000,310.17,0.0,PENDING,Not Rejected,null,claims_202402.csv,2024-02-24T00:00:00.000Z,2026-11-25T13:15:30.034Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/claims.csv


In [0]:
# Read bronze static lookup table to silver layer as they dont need any cleanup:
spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/providers").write.format("delta").mode("overwrite").save("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/providers")


In [0]:
import pyspark.sql.functions as F

df_beneficary = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/beneficiaries")
display(df_beneficary)

#Implementing SCD2 logic, thus inserting audit columns:
df_benneficiary_scd = df_beneficary \
    .withColumn("Active_flag", F.lit(True)) \
    .withColumn("Cuurent_timestamp", F.current_date()) \
    .withColumn("Last_timestamp", F.to_date(F.lit(None).cast("string")))

#writing to silver then read it and then createOrReplace a TempView.

df_benneficiary_scd.write.format("delta").mode("overwrite").save("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/beneficiary")
(spark.read.format("delta").load("dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/beneficiary")).createOrReplaceTempView("silver_beneficiary")



beneficiary_id,birth_date,gender,city,state,postal_code,region,Bronze_timestamp,Bronze_file_path
B00001,1950-08-15,M,Kolkata,WB,689113,East,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00002,1972-08-21,F,Durgapur,WB,872737,East,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00003,1950-05-31,M,Asansol,WB,281322,East,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00004,1950-12-07,M,Delhi,DL,978059,North,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00005,1991-05-02,M,Mumbai,MH,745728,West,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00006,1994-07-02,M,Pune,MH,215302,West,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00007,1997-11-18,F,Bengaluru,KA,550316,South,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00008,1968-05-08,F,Chennai,TN,934087,South,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00009,1994-03-21,M,Hyderabad,TS,462172,South,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv
B00010,1996-10-23,M,Bhubaneswar,OD,499072,East,2026-11-25T13:15:27.357Z,dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/raw/beneficiaries.csv


In [0]:
# 1. Creating a mock dataframe of incoming updates (simulating tomorrow's batch of data)
# We are changing the city for two patients to check SCD2.
updates_data = [
    ("B00001", "1950-01-01", "M", "New Tampa", "FL", "33620", "South"), # Moved to New Tampa
    ("B00002", "1965-05-15", "F", "New Austin", "TX", "73301", "South") ] # Moved to New Austin

schema = ["beneficiary_id", "birth_date", "gender", "city", "state", "postal_code", "region"]

df_updates = spark.createDataFrame(updates_data, schema)
df_updates.createOrReplaceTempView("incoming_beneficiary_updates")


# 2. Perform the SCD Type 2 MERGE using Spark SQL

spark.sql("""
MERGE INTO silver_beneficiary target
USING (
    -- SELECT 1: The incoming updated records (will be inserted as the new ACTIVE rows)
    SELECT beneficiary_id, birth_date, gender, city, state, postal_code, region, 
           'upsert' as merge_action 
    FROM incoming_beneficiary_updates
    
    UNION ALL
    
    -- SELECT 2: Join incoming with target to find existing records that need to be CLOSED
    SELECT u.beneficiary_id, u.birth_date, u.gender, u.city, u.state, u.postal_code, u.region, 
           'update' as merge_action
    FROM incoming_beneficiary_updates u
    JOIN silver_beneficiary t 
      ON u.beneficiary_id = t.beneficiary_id
    WHERE t.Active_flag = true AND u.city <> t.city
) source
ON target.beneficiary_id = source.beneficiary_id 
   AND target.Active_flag = true 
   AND source.merge_action = 'update'

-- Action 1: Close out the old record
WHEN MATCHED AND target.city <> source.city THEN
  UPDATE SET target.Active_flag = false, target.Last_timestamp = current_date()

-- Action 2: Insert the new active record
WHEN NOT MATCHED AND source.merge_action = 'upsert' THEN
  INSERT (beneficiary_id, birth_date, gender, city, state, postal_code, region, Active_flag, Cuurent_timestamp, Last_timestamp)
  VALUES (source.beneficiary_id, source.birth_date, source.gender, source.city, source.state, source.postal_code, source.region, true, current_date(), null)
""")

print("SCD Type 2 MERGE completed successfully!")


display(spark.sql("""
    SELECT beneficiary_id, city, Active_flag, Cuurent_timestamp, Last_timestamp 
    FROM silver_beneficiary 
    WHERE beneficiary_id IN ('B00001', 'B00002')
    ORDER BY beneficiary_id, Cuurent_timestamp
"""))

SCD Type 2 MERGE completed successfully!


beneficiary_id,city,Active_flag,Cuurent_timestamp,Last_timestamp
B00001,New Tampa,true,2026-11-25,null
B00001,Kolkata,false,2026-11-25,2026-11-25
B00002,Durgapur,false,2026-11-25,2026-11-25
B00002,New Austin,true,2026-11-25,null


In [0]:
#ingesting lookup tables to silver:

for lookup in ["diagnosis_codes", "procedure_codes"]:
    spark.read.format("delta").load(f"dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/bronze/{lookup}") \
        .dropDuplicates() \
        .write.format("delta").mode("overwrite").save(f"dbfs:/Volumes/workspace/default/data_pipeline_project_14_11_26/silver/{lookup}")

print("Lookup tables saved to Silver!")

Lookup tables saved to Silver!
